In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "", "catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")

catalog_name = dbutils.widgets.get("catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {catalog_name}.{config_schema_name}.dqx_rule_definitions 
    (rule_id, rule_name, rule_function, description, rule_type, rule_dimension, created_date, updated_date, argument_placeholder, is_arg_mendatory)
    VALUES 
    ('R001', 'Not Null Check', 'is_not_null', 'Ensures the column contains no null values.', 'row_level', 'Completeness', current_timestamp(), current_timestamp(), '{{"column":"<col_name>"}}', false),
    ('R002', 'Not Empty Check', 'is_not_empty', 'Ensures string columns are not empty strings.', 'column_level', 'Completeness', current_timestamp(), current_timestamp(), '{{"column":"<col_name>"}}', false),
    ('R003', 'Unique Check', 'is_unique', 'Ensures all values in the column are distinct.', 'column_level', 'Uniqueness', current_timestamp(), current_timestamp(), '{{"columns":["<col_name>"]}}', false),
    ('R004', 'Primary Key Check', 'is_primary_key', 'Combines Not Null and Unique checks.', 'column_level', 'Uniqueness / Completeness', current_timestamp(), current_timestamp(), '{{"columns": ["pk_col1", "pk_col2"]}}', true),
    ('R005', 'In Range Check', 'is_not_in_range', 'Checks whether the values in the input column are within the provided limits (inclusive of both boundaries).', 'row_level', 'Validity', current_timestamp(), current_timestamp(), '{{"column":"<col_name>","min_limit":<min>,"max_limit":<max>}}', true),
    ('R006', 'Pattern Match', 'regex_match', 'Validates column values against a specified regex pattern.', 'row_level', 'Validity', current_timestamp(), current_timestamp(), '{{"pattern": "<regex_exp>"}}', true),
    ('R007', 'Foreign Key Check', 'foreign_key', 'Ensures values exist in parent table.', 'column_level', 'Consistency / Integrity', current_timestamp(), current_timestamp(), '{{"columns":["<col_name>"],"ref_table": "catalog.schema.tablename", "ref_columns": ["<col_name>"]}}', true),
    ('R008', 'SQL Expression Check', 'sql_expression', 'Custom SQL validation.', 'column_level', 'Validity', current_timestamp(), current_timestamp(), '{{"expression": "column_a > column_b", "msg":"<message description>"}}', true),
    ('R009', 'Min Max Check', 'min_max', 'Real min/max values were used.', 'column_level', 'Validity', current_timestamp(), current_timestamp(), '{{"column":"<col_name>","min":<col_name>,"max":<col_name>}}', true),
    ('R010', 'Not Less Than Check', 'is_not_less_than', 'Checks whether the values in the input column are not less than the provided minimum.', 'row_level', 'Validity', current_timestamp(), current_timestamp(), '{{"column":"<col_name>","limit":<min>}}', true),
    ('R011', 'In Range Check', 'is_in_range', 'Checks whether the values in the input column are within the provided range (inclusive).', 'row_level', 'Validity', current_timestamp(), current_timestamp(), '{{"column":"<col_name>","min_limit":<min>,"max_limit":<max>}}', true),
    ('R012', 'Is Valid Date', 'is_valid_date', 'Ensures the column contains valid date values.', 'row_level', 'Completeness', current_timestamp(), null, '{{"column": "<col_name>"}}', false),
    ('R013', 'Is In List', 'is_in_list', 'Checks if column values are within a specified list.', 'row_level', 'Completeness', current_timestamp(), null, '{{"column": "<col_name>", "allowed": ["active", "inactive"]}}', true),
    ('R014', 'Is Not Null And Not Empty', 'is_not_null_and_not_empty', 'Ensures the column contains no null or empty values.', 'row_level', 'Completeness', current_timestamp(), null, '{{"column": "<col_name>"}}', false),
    ('R015', 'Sql Query', 'sql_query', 'Executes a custom SQL query for validation.', 'row_level', 'Completeness', current_timestamp(), null, '{{"query": "SELECT customerID, email_address, phone_number FROM customer_dimension GROUP BY customerID, email_address, phone_number HAVING COUNT(*) > 1", "msg": "Duplicate customer records"}}', true)
""")

spark.sql(f"SELECT * FROM {catalog_name}.{config_schema_name}.dqx_rule_definitions").display()